# Model Training, Implementation, and Validation



### Importing Libraries

In [53]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

### Importing Dataset -- created in Data Section

In [54]:
df = pd.read_csv("../NBADATA/SAVEDDATA/Kat_df.csv")

## Organizing Features and Target 

### Setting Features and Target

In [66]:
feature_cols = [
    "MIN_EMA_5",
    "BLK_EMA_5",
    "OPP_PACE",
    "OPP_RA_FGA"
]

target_col = "Block_Stat"

# Include SEASON only for weighting, not as a feature
model_data = df[feature_cols + [target_col, "SEASON"]].dropna().copy()

X = model_data[feature_cols]
y = model_data[target_col]

season_weights = {
    "2022-23": 0.4,
    "2023-24": 0.6,
    "2024-25": 0.8,
    "2025-26": 1.2
}

sample_weights = model_data["SEASON"].map(season_weights)

### Splitting Training and Test Data


In [67]:
split_idx = int(len(model_data) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

w_train = sample_weights.iloc[:split_idx]
w_test = sample_weights.iloc[split_idx:]

### Implementing Model

In [72]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
model.fit(X_train, y_train, logreg__sample_weight=w_train)

,steps,"[('scaler', ...), ('logreg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


## Model Evaluation

### Model Performance Metric Visualization

In [73]:


# Get predicted probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Convert probabilities to 0/1 using custom threshold
y_pred = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))

if len(set(y_test)) > 1:
    print("ROC AUC:", roc_auc_score(y_test, y_prob))
else:
    print("ROC AUC: cannot calculate because y_test only has one class")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.543859649122807
ROC AUC: 0.5359801488833748
Confusion Matrix:
[[31  0]
 [26  0]]
Classification Report:
              precision    recall  f1-score   support

           0       0.54      1.00      0.70        31
           1       0.00      0.00      0.00        26

    accuracy                           0.54        57
   macro avg       0.27      0.50      0.35        57
weighted avg       0.30      0.54      0.38        57



/opt/anaconda3/envs/parlay310/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/parlay310/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/parlay310/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [65]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# probabilities that Block_Stat = 1
y_proba = model.predict_proba(X_test)[:, 1]

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):
    y_pred_thresh = (y_proba >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred_thresh)
    precision = precision_score(y_test, y_pred_thresh, zero_division=0)
    recall = recall_score(y_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_thresh).ravel()

    threshold_results.append({
        "threshold": round(threshold, 2),
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "num_predicted_blocks": int(y_pred_thresh.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.sort_values("f1", ascending=False))

    threshold  accuracy  precision    recall        f1  true_negatives  \
1        0.15  0.473684   0.464286  1.000000  0.634146               1   
3        0.25  0.491228   0.471698  0.961538  0.632911               3   
0        0.10  0.456140   0.456140  1.000000  0.626506               0   
2        0.20  0.456140   0.454545  0.961538  0.617284               1   
4        0.30  0.561404   0.512821  0.769231  0.615385              12   
5        0.35  0.578947   0.541667  0.500000  0.520000              20   
6        0.40  0.456140   0.307692  0.153846  0.205128              22   
12       0.70  0.543860   0.000000  0.000000  0.000000              31   
15       0.85  0.543860   0.000000  0.000000  0.000000              31   
14       0.80  0.543860   0.000000  0.000000  0.000000              31   
13       0.75  0.543860   0.000000  0.000000  0.000000              31   
8        0.50  0.508772   0.000000  0.000000  0.000000              29   
11       0.65  0.543860   0.000000  0.

### Model Predicted Probability Visualization

In [39]:
results = model_data.iloc[split_idx:].copy()

results["Pred_Prob_1Plus_Block"] = y_prob
results["Predicted_025"] = y_pred

print(results[[
    "GAME_DATE",
    "OPP",
    "BLK",
    "Block_Stat",
    "Pred_Prob_1Plus_Block",
    "Predicted_025"
]].to_string())

      GAME_DATE  OPP  BLK  Block_Stat  Pred_Prob_1Plus_Block  Predicted_025
232  2025-12-31  SAS    0           0               0.405953              1
233  2026-01-03  PHI    0           0               0.446686              1
234  2026-01-05  DET    0           0               0.515387              1
236  2026-01-09  PHX    0           0               0.274962              0
237  2026-01-11  POR    2           1               0.414771              1
238  2026-01-14  SAC    0           0               0.244175              0
239  2026-01-15  GSW    1           1               0.315548              1
240  2026-01-17  PHX    1           1               0.304483              1
241  2026-01-19  DAL    1           1               0.387692              1
242  2026-01-21  BKN    0           0               0.403635              1
243  2026-01-24  PHI    1           1               0.350263              1
244  2026-01-27  SAC    0           0               0.139141              0
245  2026-01

### Model Accuracy When Predicted Prob > 0.4

In [41]:
high_conf = results[results["Pred_Prob_1Plus_Block"] >= 0.40]

print(high_conf[[
    "GAME_DATE",
    "OPP",
    "BLK",
    "Block_Stat",
    "Pred_Prob_1Plus_Block"
]].to_string())

print("High confidence sample size:", len(high_conf))
print("Hit rate when prob >= 0.40:", high_conf["Block_Stat"].mean())

      GAME_DATE  OPP  BLK  Block_Stat  Pred_Prob_1Plus_Block
232  2025-12-31  SAS    0           0               0.405953
233  2026-01-03  PHI    0           0               0.446686
234  2026-01-05  DET    0           0               0.515387
237  2026-01-11  POR    2           1               0.414771
242  2026-01-21  BKN    0           0               0.403635
252  2026-02-11  PHI    1           1               0.410766
253  2026-02-19  DET    0           0               0.419644
255  2026-02-22  CHI    0           0               0.472624
267  2026-03-20  BKN    0           0               0.405840
269  2026-03-24  NOP    0           0               0.512036
279  2026-04-23  ATL    2           1               0.418092
280  2026-04-25  ATL    0           0               0.437084
282  2026-04-30  ATL    1           1               0.423540
High confidence sample size: 13
Hit rate when prob >= 0.40: 0.3076923076923077


In [19]:
low_conf = results[results["Pred_Prob_1Plus_Block"] < 0.25]

medium_conf = results[
    (results["Pred_Prob_1Plus_Block"] >= 0.25) &
    (results["Pred_Prob_1Plus_Block"] < 0.40)
]

high_conf = results[results["Pred_Prob_1Plus_Block"] >= 0.40]

print("Low sample:", len(low_conf), "hit rate:", low_conf["Block_Stat"].mean())
print("Medium sample:", len(medium_conf), "hit rate:", medium_conf["Block_Stat"].mean())
print("High sample:", len(high_conf), "hit rate:", high_conf["Block_Stat"].mean())

Low sample: 3 hit rate: 0.3333333333333333
Medium sample: 11 hit rate: 0.7272727272727273
High sample: 4 hit rate: 1.0


In [21]:
import joblib

joblib.dump(model, "kat_block_model.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")
df.to_csv("kat_features.csv", index=False)